# Postprocess PINN calculations

This notebook postprocesses the calculations of the Physics-Informed Neural Network (PINN) for both empirical and simulated dust flux depositions during the Holocene and the Last Glacial Maximum. The results are visualized on a global map and several statistics are compared with kriging interpolation.

## Preliminaries

Import the necessary libraries and specify the data folders.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
FIGURE_PATH = "../Figures/"
DATA_PATH = "../Data/"
DATASETS_PATH = DATA_PATH + "processed_data/"
MODEL_RESULTS_PATH = DATA_PATH + "model_results/"

In [ ]:
with open("functions_plot_calculations.py", 'r', encoding='utf-8') as file:
    content = file.read()

# Execute the content of the .py file
exec(content)

# Ensure JCP style is applied after exec
from style_jcp import (apply_style, SAVE_DPI, DIFFUSIVITY_VLIM, SCATTER_AXIS_LIM,
                        CMAP_DIFFUSIVITY, CMAP_DIVERGING)
apply_style()

In [ ]:
from style_jcp import load_world
world = load_world()

In [ ]:
df_empirical_Holocene = pd.read_csv(DATASETS_PATH + "df_empirical_Holocene.csv")
df_empirical_LGM = pd.read_csv(DATASETS_PATH + "df_empirical_LGM.csv")
df_simulated_Holocene = pd.read_csv(DATASETS_PATH + "df_simulation_Holocene.csv")
df_simulated_LGM = pd.read_csv(DATASETS_PATH + "df_simulation_LGM.csv")

In [ ]:
df_kriging_Holocene = pd.read_csv(DATASETS_PATH + "df_kriging_Holocene.csv")
df_kriging_LGM = pd.read_csv(DATASETS_PATH + "df_kriging_LGM.csv")

df_kriging_Holocene.sort_values(['lat', 'lon'], ascending=True, inplace=True)
df_kriging_LGM.sort_values(['lat', 'lon'], ascending=True, inplace=True)

In [ ]:
name_period = "Holocene"
name_dataset = "empirical"

In [ ]:
df_pinn_results_training = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+"_training_points.csv")
df_pinn_results_global_grid = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+"_global_grid.csv")

In [ ]:
from functions_plot_calculations import plot_dust_deposition_map

plot_dust_deposition_map(
    df_pinn_results_global_grid,
    df_pinn_results_training,
    title='PINN calculation for Holocene',
    name_to_save='PINN_MAP_EMPIRICAL_HOLOCENE',
    figure_save_path=FIGURE_PATH,
    label_str='log_dep',
)

In [ ]:
from functions_plot_calculations import plot_dust_deposition_map_zoom

plot_dust_deposition_map_zoom(
    df=df_pinn_results_global_grid,
    title='PINN calculation for Holocene',
    name_to_save='PINN_ZOOM_EMPIRICAL_HOLOCENE',
    figure_save_path=FIGURE_PATH,
    label_str='PINN_log_dep',
)

plot_dust_deposition_map_zoom(
    df=df_kriging_Holocene,
    title='Kriging interpolation for Holocene',
    name_to_save='KRIGING_ZOOM_EMPIRICAL_HOLOCENE',
    figure_save_path=FIGURE_PATH,
    label_str='log_dep',
)

In [ ]:
from scipy.interpolate import griddata

x = df_kriging_Holocene[['lon', 'lat']].values
y = df_kriging_Holocene['log_dep'].values
x_interpolate = df_pinn_results_training[['lon', 'lat']].values
y_interpolate = griddata(x, y, x_interpolate, method='nearest')

df_pinn_results_training['kriging_log_dep'] = y_interpolate

In [ ]:
from functions_plot_calculations import plot_scatterplot_empirical

plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="log_dep",
    y_variable="PINN_log_dep",
    x_label="Empirical dataset",
    y_label="PINN calculation",
    name_to_save="SCATTER_PINN_EMPIRICAL_HOLOCENE",
    figure_save_path=FIGURE_PATH,
)

plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="log_dep",
    y_variable="kriging_log_dep",
    x_label="Empirical dataset",
    y_label="Kriging interpolation",
    name_to_save="SCATTER_KRIGING_EMPIRICAL_HOLOCENE",
    figure_save_path=FIGURE_PATH,
)

plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="PINN_log_dep",
    y_variable="kriging_log_dep",
    x_label="PINN calculation",
    y_label="Kriging interpolation",
    name_to_save="SCATTER_PINN_KRIGING_HOLOCENE",
    figure_save_path=FIGURE_PATH,
)

In [ ]:
from functions_plot_calculations import plot_hist

plot_hist(
    df=df_pinn_results_global_grid,
    title='PINN calculation for Holocene',
    name_to_save='PINN_HIST_EMPIRICAL_HOLOCENE',
    figure_save_path=FIGURE_PATH,
)

In [ ]:
def plot_variable_map(df_grid, df_points, variable_name, title, name_to_save, figure_save_path, cmap='viridis', vmin=None, vmax=None, label_cbar='Value'):
    try:
        data_reshape = df_grid[variable_name].values.reshape(60, 120)
    except ValueError:
        n_lat = len(df_grid['lat'].unique())
        n_lon = len(df_grid['lon'].unique())
        data_reshape = df_grid[variable_name].values.reshape(n_lat, n_lon)

    fig, ax = plt.subplots(figsize=(7.48, 5.0))

    if vmin is None: vmin = df_grid[variable_name].min()
    if vmax is None: vmax = df_grid[variable_name].max()

    h = ax.imshow(data_reshape,
                  origin='lower',
                  extent=[-180, 180, -90, 90],
                  cmap=cmap,
                  vmin=vmin,
                  vmax=vmax)

    world.dissolve(by='continent').boundary.plot(ax=ax, color='black', linewidth=0.8)

    ax.set(xlabel='Longitude', ylabel='Latitude', title=title)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xticks(np.arange(-180, 181, 45))
    ax.set_yticks(np.arange(-90, 91, 30))

    cbar = plt.colorbar(h, ax=ax, orientation='horizontal', pad=0.08, fraction=0.05)
    cbar.set_label(label_cbar)

    plt.savefig(f"{figure_save_path}/{name_to_save}.pdf", bbox_inches='tight', dpi=SAVE_DPI)
    plt.savefig(f"{figure_save_path}/{name_to_save}.png", bbox_inches='tight', dpi=SAVE_DPI)
    plt.show()

In [ ]:
plot_variable_map(
    df_grid=df_pinn_results_global_grid,
    df_points=None,
    variable_name='PINN_Diffusivity',
    title='Inferred Diffusivity Field $D(x)$ (Holocene)',
    name_to_save='PINN_DIFFUSIVITY_MAP_EMPIRICAL_HOLOCENE',
    figure_save_path=FIGURE_PATH,
    cmap=CMAP_DIFFUSIVITY,
    vmin=DIFFUSIVITY_VLIM[0],
    vmax=DIFFUSIVITY_VLIM[1],
    label_cbar='Diffusivity Coeff [-]'
)

In [ ]:
name_period = "LGM"
name_dataset = "empirical"

In [ ]:
df_pinn_results_training = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+"_training_points.csv")
df_pinn_results_global_grid = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+"_global_grid.csv")

In [ ]:
plot_dust_deposition_map(
    df_pinn_results_global_grid,
    df_pinn_results_training,
    title='PINN calculation for LGM',
    name_to_save='PINN_MAP_EMPIRICAL_LGM',
    figure_save_path=FIGURE_PATH,
    label_str='log_dep',
)

In [ ]:
plot_dust_deposition_map_zoom(
    df=df_pinn_results_global_grid,
    title='PINN calculation for LGM',
    name_to_save='PINN_ZOOM_EMPIRICAL_LGM',
    figure_save_path=FIGURE_PATH,
    label_str='PINN_log_dep',
)

plot_dust_deposition_map_zoom(
    df=df_kriging_LGM,
    title='Kriging interpolation for LGM',
    name_to_save='KRIGING_ZOOM_EMPIRICAL_LGM',
    figure_save_path=FIGURE_PATH,
    label_str='log_dep',
)

In [ ]:
x = df_kriging_LGM[['lon', 'lat']].values
y = df_kriging_LGM['log_dep'].values
x_interpolate = df_pinn_results_training[['lon', 'lat']].values
y_interpolate = griddata(x, y, x_interpolate, method='nearest')

df_pinn_results_training['kriging_log_dep'] = y_interpolate

In [ ]:
plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="log_dep",
    y_variable="PINN_log_dep",
    x_label="Empirical dataset",
    y_label="PINN calculation",
    name_to_save="SCATTER_PINN_EMPIRICAL_LGM",
    figure_save_path=FIGURE_PATH,
)

plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="log_dep",
    y_variable="kriging_log_dep",
    x_label="Empirical dataset",
    y_label="Kriging interpolation",
    name_to_save="SCATTER_KRIGING_EMPIRICAL_LGM",
    figure_save_path=FIGURE_PATH,
)

plot_scatterplot_empirical(
    df=df_pinn_results_training,
    x_variable="PINN_log_dep",
    y_variable="kriging_log_dep",
    x_label="PINN calculation",
    y_label="Kriging interpolation",
    name_to_save="SCATTER_PINN_KRIGING_LGM",
    figure_save_path=FIGURE_PATH,
)

In [ ]:
plot_hist(
    df=df_pinn_results_global_grid,
    title='PINN calculation for LGM',
    name_to_save='PINN_HIST_EMPIRICAL_LGM',
    figure_save_path=FIGURE_PATH,
)

In [ ]:
from functions_plot_calculations import plot_spatial_error_map

df_pinn_results_training['abs_error'] = np.abs(df_pinn_results_training['log_dep'] - df_pinn_results_training['PINN_log_dep'])

plot_spatial_error_map(
    df=df_pinn_results_training,
    title=f"Absolute Error Distribution ({name_period} - {name_dataset})",
    name_to_save=f"ERROR_MAP_{name_dataset.upper()}_{name_period.upper()}",
    figure_save_path=FIGURE_PATH,
    label_str='abs_error',
    measure_units='Absolute Error |log10(g m-2 a-1)|'
)

In [ ]:
plot_variable_map(
    df_grid=df_pinn_results_global_grid,
    df_points=None,
    variable_name='PINN_Diffusivity',
    title='Inferred Diffusivity Field $D(x)$ (LGM)',
    name_to_save='PINN_DIFFUSIVITY_MAP_EMPIRICAL_LGM',
    figure_save_path=FIGURE_PATH,
    cmap=CMAP_DIFFUSIVITY,
    vmin=DIFFUSIVITY_VLIM[0],
    vmax=DIFFUSIVITY_VLIM[1],
    label_cbar='Diffusivity Coeff [-]'
)

In [ ]:
name_period = "Holocene"
name_dataset = "simulated"

In [ ]:
df_pinn_results = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+".csv")

In [ ]:
from functions_plot_calculations import plot_dust_deposition_simulated

plot_dust_deposition_simulated(
    df=df_pinn_results,
    title='Holocene',
    name_to_save='PINN_MAP_SIMULATED_HOLOCENE',
    figure_save_path=FIGURE_PATH,
)

In [ ]:
from functions_plot_calculations import plot_scatterplot_simulated

plot_scatterplot_simulated(
    df=df_pinn_results,
    name_to_save="SCATTER_SIMULATED_HOLOCENE",
    x_label="Simulated data",
    y_label="PINN calculation",
    figure_save_path=FIGURE_PATH,
)

In [ ]:
name_period = "LGM"
name_dataset = "simulated"

In [ ]:
df_pinn_results = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_"+name_dataset+"_"+name_period+".csv")

In [ ]:
plot_dust_deposition_simulated(
    df=df_pinn_results,
    title='Last Glacial Maximum',
    name_to_save='PINN_MAP_SIMULATED_LGM',
    figure_save_path=FIGURE_PATH,
)

In [ ]:
plot_scatterplot_simulated(
    df=df_pinn_results,
    name_to_save="SCATTER_SIMULATED_LGM",
    x_label="Simulated data",
    y_label="PINN calculation",
    figure_save_path=FIGURE_PATH,
)

In [ ]:
import cartopy.crs as ccrs

plt.figure(figsize=(7.48, 4.0))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(linewidth=0.8, color='black')

sc = ax.scatter(df_pinn_results['lon'], df_pinn_results['lat'],
                c=df_pinn_results['PINN_Diffusivity'],
                transform=ccrs.PlateCarree(),
                cmap=CMAP_DIFFUSIVITY,
                vmin=DIFFUSIVITY_VLIM[0],
                vmax=DIFFUSIVITY_VLIM[1],
                s=1)

cbar = plt.colorbar(sc, label='Inferred Diffusivity Coeff $D$ [-]',
                    orientation='horizontal', pad=0.05, fraction=0.05)
plt.title(f'Inferred Diffusivity $D(x)$ (Simulated LGM, Mean D={df_pinn_results["PINN_Diffusivity"].mean():.4f})')

save_name = "PINN_DIFFUSIVITY_MAP_SIMULATED_LGM"
plt.savefig(FIGURE_PATH + f"{save_name}.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.savefig(FIGURE_PATH + f"{save_name}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import cartopy.crs as ccrs
import pandas as pd

filename_sim_hol = MODEL_RESULTS_PATH + "df_pinn_simulated_Holocene.csv"
try:
    df_sim_hol = pd.read_csv(filename_sim_hol)
except FileNotFoundError:
    print(f"Error: file not found {filename_sim_hol}")

plt.figure(figsize=(7.48, 4.0))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(linewidth=0.8, color='black')

sc = ax.scatter(df_sim_hol['lon'], df_sim_hol['lat'],
                c=df_sim_hol['PINN_Diffusivity'],
                transform=ccrs.PlateCarree(),
                cmap=CMAP_DIFFUSIVITY,
                vmin=DIFFUSIVITY_VLIM[0],
                vmax=DIFFUSIVITY_VLIM[1],
                s=1)

cbar = plt.colorbar(sc, label='Inferred Diffusivity Coeff $D$ [-]',
                    orientation='horizontal', pad=0.05, fraction=0.05)
plt.title(f'Inferred Diffusivity $D(x)$ (Simulated Holocene, Mean D={df_sim_hol["PINN_Diffusivity"].mean():.4f})')

save_name = "PINN_DIFFUSIVITY_MAP_SIMULATED_HOLOCENE"
plt.savefig(FIGURE_PATH + f"{save_name}.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.savefig(FIGURE_PATH + f"{save_name}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df_ours = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_simulated_LGM.csv")
df_base = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_simulated_LGM_Baseline.csv")

rmse_base = np.sqrt(mean_squared_error(df_base['log_dep'], df_base['PINN_log_dep']))
mae_base = mean_absolute_error(df_base['log_dep'], df_base['PINN_log_dep'])
r2_base = r2_score(df_base['log_dep'], df_base['PINN_log_dep'])

rmse_ours = np.sqrt(mean_squared_error(df_ours['log_dep'], df_ours['PINN_log_dep']))
mae_ours = mean_absolute_error(df_ours['log_dep'], df_ours['PINN_log_dep'])
r2_ours = r2_score(df_ours['log_dep'], df_ours['PINN_log_dep'])

fig, axes = plt.subplots(1, 2, figsize=(7.48, 3.5))

axes[0].scatter(df_base['log_dep'], df_base['PINN_log_dep'], alpha=0.5, c='gray', s=2)
axes[0].plot(SCATTER_AXIS_LIM, SCATTER_AXIS_LIM, 'k--', lw=1.0)
axes[0].set_xlim(*SCATTER_AXIS_LIM)
axes[0].set_ylim(*SCATTER_AXIS_LIM)
axes[0].set_aspect('equal')
axes[0].set_title(f"Baseline (No Transfer)\nRMSE={rmse_base:.2f}, MAE={mae_base:.2f}, $R^2$={r2_base:.2f}")
axes[0].set_xlabel("Simulated Truth")
axes[0].set_ylabel("PINN Prediction")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df_ours['log_dep'], df_ours['PINN_log_dep'], alpha=0.5, c='red', s=2)
axes[1].plot(SCATTER_AXIS_LIM, SCATTER_AXIS_LIM, 'k--', lw=1.0)
axes[1].set_xlim(*SCATTER_AXIS_LIM)
axes[1].set_ylim(*SCATTER_AXIS_LIM)
axes[1].set_aspect('equal')
axes[1].set_title(f"Ours (Transfer Learning)\nRMSE={rmse_ours:.2f}, MAE={mae_ours:.2f}, $R^2$={r2_ours:.2f}")
axes[1].set_xlabel("Simulated Truth")
axes[1].set_ylabel("PINN Prediction")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_PATH + "INNOVATION_SCATTER_COMPARISON_SIMULATED_LGM.png", dpi=SAVE_DPI)
plt.savefig(FIGURE_PATH + "INNOVATION_SCATTER_COMPARISON_SIMULATED_LGM.pdf")
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df_ours_scatter = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_LGM_training_points.csv")
df_base_scatter = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_LGM_Baseline_training_points.csv")

rmse_base = np.sqrt(mean_squared_error(df_base_scatter['log_dep'], df_base_scatter['PINN_log_dep']))
r2_base = r2_score(df_base_scatter['log_dep'], df_base_scatter['PINN_log_dep'])
mae_base = mean_absolute_error(df_base_scatter['log_dep'], df_base_scatter['PINN_log_dep'])

rmse_ours = np.sqrt(mean_squared_error(df_ours_scatter['log_dep'], df_ours_scatter['PINN_log_dep']))
r2_ours = r2_score(df_ours_scatter['log_dep'], df_ours_scatter['PINN_log_dep'])
mae_ours = mean_absolute_error(df_ours_scatter['log_dep'], df_ours_scatter['PINN_log_dep'])

fig, axes = plt.subplots(1, 2, figsize=(7.48, 3.5))

axes[0].scatter(df_base_scatter['log_dep'], df_base_scatter['PINN_log_dep'], alpha=0.6, c='gray', edgecolors='k', linewidths=0.3, s=15)
axes[0].plot(SCATTER_AXIS_LIM, SCATTER_AXIS_LIM, 'k--', lw=1.0)
axes[0].set_xlim(*SCATTER_AXIS_LIM)
axes[0].set_ylim(*SCATTER_AXIS_LIM)
axes[0].set_aspect('equal')
axes[0].set_title(f"Baseline (No Transfer)\nRMSE={rmse_base:.3f}, MAE={mae_base:.3f}, $R^2$={r2_base:.3f}")
axes[0].set_xlabel("Observed Data (DIRTMAP)")
axes[0].set_ylabel("PINN Prediction")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df_ours_scatter['log_dep'], df_ours_scatter['PINN_log_dep'], alpha=0.6, c='red', edgecolors='k', linewidths=0.3, s=15)
axes[1].plot(SCATTER_AXIS_LIM, SCATTER_AXIS_LIM, 'k--', lw=1.0)
axes[1].set_xlim(*SCATTER_AXIS_LIM)
axes[1].set_ylim(*SCATTER_AXIS_LIM)
axes[1].set_aspect('equal')
axes[1].set_title(f"Ours (Transfer Learning)\nRMSE={rmse_ours:.3f}, MAE={mae_ours:.3f}, $R^2$={r2_ours:.3f}")
axes[1].set_xlabel("Observed Data (DIRTMAP)")
axes[1].set_ylabel("PINN Prediction")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURE_PATH + "INNOVATION_SCATTER_COMPARISON_EMPIRICAL_LGM.png", dpi=SAVE_DPI)
plt.savefig(FIGURE_PATH + "INNOVATION_SCATTER_COMPARISON_EMPIRICAL_LGM.pdf")
plt.show()

In [ ]:
import cartopy.crs as ccrs

df_ours_map = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_LGM_global_grid.csv")
df_base_map = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_LGM_Baseline_global_grid.csv")

diff_values = df_ours_map['PINN_log_dep'] - df_base_map['PINN_log_dep']
max_diff = diff_values.abs().max()
mean_diff = diff_values.abs().mean()

if max_diff < 1e-6:
    print("Warning: data identical, check Baseline output.")
else:
    fig = plt.figure(figsize=(7.48, 4.0))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.5)

    limit = max(abs(diff_values.min()), abs(diff_values.max()))

    sc = ax.scatter(df_ours_map['lon'], df_ours_map['lat'],
                    c=diff_values,
                    transform=ccrs.PlateCarree(),
                    cmap=CMAP_DIVERGING,
                    vmin=-limit, vmax=limit,
                    s=1)

    plt.title("Difference Map: Transfer $-$ Baseline (Empirical LGM)")

    cbar = plt.colorbar(sc, orientation='horizontal', fraction=0.05, pad=0.05)
    cbar.set_label('Log Dust Flux Difference (Transfer $-$ Baseline)')

    plt.savefig(FIGURE_PATH + "INNOVATION_DIFFERENCE_MAP_EMPIRICAL_LGM.png", dpi=SAVE_DPI, bbox_inches='tight')
    plt.savefig(FIGURE_PATH + "INNOVATION_DIFFERENCE_MAP_EMPIRICAL_LGM.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
import cartopy.crs as ccrs
import pandas as pd
import numpy as np

df_ours_sim = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_simulated_LGM.csv")
df_base_sim = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_simulated_LGM_Baseline.csv")

if len(df_ours_sim) != len(df_base_sim):
    print(f"Warning: row count mismatch. Ours: {len(df_ours_sim)}, Baseline: {len(df_base_sim)}")
    min_len = min(len(df_ours_sim), len(df_base_sim))
    df_ours_sim = df_ours_sim.iloc[:min_len]
    df_base_sim = df_base_sim.iloc[:min_len]

diff_values = df_ours_sim['PINN_log_dep'] - df_base_sim['PINN_log_dep']
max_diff = diff_values.abs().max()
mean_diff = diff_values.abs().mean()

fig = plt.figure(figsize=(7.48, 4.0))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(linewidth=0.5)

limit = max(abs(diff_values.min()), abs(diff_values.max()))

sc = ax.scatter(df_ours_sim['lon'], df_ours_sim['lat'],
                c=diff_values,
                transform=ccrs.PlateCarree(),
                cmap=CMAP_DIVERGING,
                vmin=-limit, vmax=limit,
                s=2)

plt.title("Difference Map: Transfer $-$ Baseline (Simulated LGM)")

cbar = plt.colorbar(sc, orientation='horizontal', fraction=0.05, pad=0.05)
cbar.set_label('Log Dust Flux Difference (Transfer $-$ Baseline)')

plt.savefig(FIGURE_PATH + "INNOVATION_DIFFERENCE_MAP_SIMULATED_LGM.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.savefig(FIGURE_PATH + "INNOVATION_DIFFERENCE_MAP_SIMULATED_LGM.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_hol = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_Holocene_global_grid.csv")
df_lgm = pd.read_csv(MODEL_RESULTS_PATH + "df_pinn_empirical_LGM_global_grid.csv")

D_hol = df_hol['PINN_Diffusivity']
D_lgm = df_lgm['PINN_Diffusivity']

mean_hol = D_hol.mean()
mean_lgm = D_lgm.mean()
change_ratio = (mean_lgm - mean_hol) / mean_hol * 100

plt.figure(figsize=(7.48, 4.0))

plt.hist(D_hol, bins=50, alpha=0.6, label=f'Holocene (Mean={mean_hol:.4f})', color='#3b4cc0', density=True)
plt.hist(D_lgm, bins=50, alpha=0.6, label=f'LGM (Mean={mean_lgm:.4f})', color='#b40426', density=True)

plt.title("Distribution of Inferred Diffusivity Coefficient $D(x)$")
plt.xlabel("Diffusivity Value")
plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)

text_str = f"Change Ratio: {change_ratio:.2f}%"
plt.text(0.5, 0.5, text_str, transform=plt.gca().transAxes,
         fontweight='bold',
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.savefig(FIGURE_PATH + "DIFFUSIVITY_DISTRIBUTION_COMPARISON.png", dpi=SAVE_DPI)
plt.savefig(FIGURE_PATH + "DIFFUSIVITY_DISTRIBUTION_COMPARISON.pdf")
plt.show()